# Leakage-aware Evaluation of Infant Cry Classification — Colab runner

Runs the three core experiments of the paper (`multiclass`, `binary`, `leakage`) from Google Colab, with data and repo living on Google Drive. No local GPU is required — the pipeline falls back to CPU/scikit-learn automatically (see the last cell).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the repo

Two options — pick ONE. Default below assumes the repo already lives inside your Drive (e.g. you cloned or copied it there once, alongside the dataset). Use the commented block instead if you'd rather clone fresh into the Colab runtime each session.

In [ ]:
# Option A (default): repo already present in Drive.
REPO_DIR = "/content/drive/MyDrive/leakage-aware-infant-cry-classification"
%cd $REPO_DIR

# Option B: clone fresh into the Colab runtime instead (uncomment and set REPO_URL).
# REPO_URL = "https://github.com/<user>/leakage-aware-infant-cry-classification.git"
# REPO_DIR = "/content/leakage-aware-infant-cry-classification"
# !git clone $REPO_URL $REPO_DIR
# %cd $REPO_DIR

## 3. Point to the dataset

See [`data/README.md`](../data/README.md) for the expected folder layout (`1s_asphyxia/`, `1s_deaf/`, `1s_hunger/`, `1s_normal/`, `1s_pain/`). The dataset itself is not distributed in this repo — request it from the Baby Chillanto custodians and place it in your Drive.

In [ ]:
import os

DATA_DIR = "/content/drive/MyDrive/leakage-aware-infant-cry-classification/data/dataset"

expected = ["1s_asphyxia", "1s_deaf", "1s_hunger", "1s_normal", "1s_pain"]
missing = [d for d in expected if not os.path.isdir(os.path.join(DATA_DIR, d))]
assert not missing, (
    f"DATA_DIR '{DATA_DIR}' is missing expected subfolders: {missing}. "
    "Place the Baby Chillanto 1s_* folders there before continuing "
    "(see data/README.md)."
)
print("Dataset layout OK:", DATA_DIR)

## 4. Install dependencies

In [ ]:
%pip install -r requirements.txt

## 5. Run the three experiments

Flags shown are the scripts' own defaults (100 Optuna trials, 5 seeds, 5-fold CV, `k_best_d=60`, `fv_n_components=16`) — the same ones used to produce `results/*.csv` in this repo. Lower `--n_trials`/`--n_seeds` for a quick smoke test before committing to a full run.

In [ ]:
# Multiclass (Tables 2-3): feature-set x classifier Optuna search
!python main.py multiclass --data_dir "$DATA_DIR" --n_trials 100 --n_seeds 5 --n_splits 5 --balanced \
    --csv_summary results/optuna_summary_multiclass.csv --csv_history results/optuna_history_multiclass.csv

In [ ]:
# Binary (Table 4): healthy vs pathology
!python main.py binary --data_dir "$DATA_DIR" --n_trials 100 --n_seeds 5 --n_splits 5 --balanced \
    --csv_summary results/binary_summary.csv --csv_history results/binary_history.csv

In [ ]:
# Leakage comparison (Section 4.3, Tables 5-6, Figures 6-8): S-1s vs G-1s
!python main.py leakage --data_dir "$DATA_DIR" --n_seeds 5 --n_splits 5 --balanced \
    --csv_results results/leakage_results.csv --csv_stats results/leakage_stats.csv --plot_dir results/leakage_plots

## Note on compute backend

`src/gpu_classifiers.py` tries `cuML` (RAPIDS) and PyTorch+CUDA for the SVM/kNN/MLP classifiers, and **falls back to scikit-learn on CPU automatically** if they aren't available — this CPU/scikit-learn fallback is what reproduces the exact figures reported in the paper. A Colab GPU runtime will use PyTorch for the MLP if CUDA is detected (Colab ships PyTorch already); installing `cuml-cu12` is optional and only speeds up SVM/kNN, it does not change what numbers you get from the MLP/RandomForest/etc. paths. `cuML` has no native Windows build (Linux/WSL2 or Colab only).